# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaifLatki/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

My rule is: rank pages by visible traffic and stale-but-fixable opportunity, then boost items that are near page one, low CTR, or already declining with demand. The goal is to surface refresh candidates that are visible enough to matter and stale enough to deserve attention, without using hidden product flags.

The rule emits a small set of transparent reason codes: stale_visible_page, declining_with_demand, thin_visible_page, page_one_decay_risk, low_ctr_visible_page, low_engagement_visible_page, and general_refresh_review.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

cwd = Path.cwd().resolve()
repo = None
for candidate in [cwd, *cwd.parents]:
    feature_path = candidate / "data" / "processed" / "refresh_feature_vector.csv"
    if feature_path.exists():
        repo = candidate
        break
if repo is None:
    raise FileNotFoundError("Could not find the project root for the feature vector.")

feature_df = pd.read_csv(repo / "data" / "processed" / "refresh_feature_vector.csv")
feature_df["is_declining_label"] = feature_df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Prepared rows: {len(feature_df):,}")
print(f"Declining rate: {feature_df['is_declining_label'].mean():.3f}")
print(feature_df[["content_id", "client_id", "impressions_90d", "days_since_last_update", "avg_position", "word_count", "ctr", "engagement_rate", "trend_direction"]].head(5).to_string(index=False))


Prepared rows: 30,000
Declining rate: 0.542
          content_id         client_id  impressions_90d  days_since_last_update  avg_position  word_count  ctr  engagement_rate trend_direction
content_304f48230142 client_f369cb89fc             3803                      20          10.6      3221.0 0.76             5.88            down
content_a1fb4e703a9e client_4e07408562            15320                      25          20.3      2481.0 0.05             0.00            down
content_9aa793d4d895 client_7f2253d7e2            12581                      20          36.5      3515.0 0.09             0.00            down
content_331d6c4de07b client_19581e27de            11751                      22           6.2         0.0 0.49             1.28          stable
content_d99b7a2d90ca client_3fdba35f04            19140                      14          44.0      2803.0 0.13             0.00            down


## 2. Build the ranked queue (writes the CSV)

The score below is a transparent weighted heuristic: visibility matters most, then freshness risk, then page-one opportunity, then depth gap. It is deliberately explainable and frozen for comparison against any trained model later.

The code writes the ranked queue to the processed dataset and also mirrors it into the outputs folder so the notebook is easy to review in the repo.


In [2]:
def percentile_rank(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    return values.rank(method="average", pct=True).fillna(0)


def normalize(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    minimum = values.min()
    maximum = values.max()
    if not np.isfinite(minimum) or not np.isfinite(maximum) or maximum == minimum:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - minimum) / (maximum - minimum)


def reason_codes(row: pd.Series) -> list[str]:
    reasons: list[str] = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if 0 < row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and ((0 < row["engagement_rate"] < 30) or (0 < row["scroll_rate"] < 30)):
        reasons.append("low_engagement_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return reasons


def suggested_action(row: pd.Series) -> str:
    reasons = set(str(row["reason_codes"]).split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"
    return "monitor"


df = feature_df.copy()
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"])) * df["visibility_score"]
df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

df["reason_codes"] = df.apply(lambda row: "|".join(reason_codes(row)), axis=1)
df["suggested_action_baseline"] = df.apply(suggested_action, axis=1)
df["baseline_rank"] = df["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_refresh_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "suggested_action_baseline",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction",
]

ranked = df[output_columns].sort_values("baseline_rank").reset_index(drop=True)
processed_path = repo / "data" / "processed" / "baseline_refresh_queue.csv"
output_path = repo / "outputs" / "baseline_action_score.csv"
processed_path.parent.mkdir(parents=True, exist_ok=True)
output_path.parent.mkdir(parents=True, exist_ok=True)
ranked.to_csv(processed_path, index=False)
ranked.to_csv(output_path, index=False)

print(f"Wrote processed queue to {processed_path}")
print(f"Wrote review queue to {output_path}")
print(f"Top score: {ranked['baseline_refresh_score'].max():.3f}")
print(f"Top-50 declining rate: {ranked.head(50)['is_declining_label'].mean():.3f}")
print(ranked.head(10).to_string(index=False))


Wrote processed queue to C:\Users\hp\Desktop\flyrank-ml-internship\data\processed\baseline_refresh_queue.csv
Wrote review queue to C:\Users\hp\Desktop\flyrank-ml-internship\outputs\baseline_action_score.csv
Top score: 0.941
Top-50 declining rate: 0.340
          content_id         client_id  baseline_rank  baseline_refresh_score  visibility_score  freshness_risk_score  position_opportunity_score  depth_gap_score                                                                               reason_codes suggested_action_baseline  is_declining_label  impressions_90d  clicks_90d  sessions_90d  avg_position  ctr  engagement_rate  scroll_rate  content_age_days  days_since_last_update  word_count trend_direction
content_9532f197bbc8 client_4e07408562              1                0.941189          0.999633                0.8432                    0.979233         0.871347                      declining_with_demand|page_one_decay_risk|low_engagement_visible_page                   refresh      

## 3. Top-20 review

The top of the queue is where the logic becomes visible to a human. I reviewed the highest ranks for actionability, reason-coding clarity, and obvious failure modes. A high score is only trustworthy if the pages are understandable and the reasons hold.

Most of the top ranks are visible, stale, and on page one with weak engagement or low CTR. The weak cases are the ones that appear high because they are large and stale, but they may still be fine pages that merely need a refresh or a small title revision rather than an urgent content action.


In [3]:
review = ranked.head(20).copy()
review["confidence_note"] = np.where(
    review["baseline_refresh_score"] >= 0.90,
    "High-confidence refresh opportunity: high visibility, stale page, and the page-one/CTR pattern is consistent with a real action signal.",
    "Moderate-confidence candidate: visible and stale, but may be a healthy page that is simply not converting or needs a narrower UX fix.",
)
review["what_would_make_it_wrong"] = (
    "A good page that is already strong on engagement would lower the score; a title or position artifact could also exaggerate the page-one opportunity."
)
print(review[[
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_refresh_score",
    "suggested_action_baseline",
    "reason_codes",
    "confidence_note",
    "what_would_make_it_wrong",
]].to_string(index=False))


 baseline_rank           content_id         client_id  baseline_refresh_score suggested_action_baseline                                                                               reason_codes                                                                                                                         confidence_note                                                                                                                             what_would_make_it_wrong
             1 content_9532f197bbc8 client_4e07408562                0.941189                   refresh                      declining_with_demand|page_one_decay_risk|low_engagement_visible_page High-confidence refresh opportunity: high visibility, stale page, and the page-one/CTR pattern is consistent with a real action signal. A good page that is already strong on engagement would lower the score; a title or position artifact could also exaggerate the page-one opportunity.
             2 content_4d1fe5b32dc2 clie

## 4. Weak picks + leakage check

The weakest picks are the ones that score highly because they are large and stale, but they are not actually failing or declining. I check the top list for pages where the score looks too good because of scale alone or because the signal is simply a rank artifact instead of a real content issue.

The model is leakage-safe because it uses only already-measured 90-day traffic, freshness, position, and content depth — no product decision flags, no hidden labels, and no future-window variables.


In [4]:
top50 = ranked.head(50).copy()
weak_picks = top50[top50["is_declining_label"] == 0].head(10)
print("Top-50 declining rate:", round(top50["is_declining_label"].mean(), 3))
print("Top-50 stable pages:", int((top50["is_declining_label"] == 0).sum()))
print("\nWeakest high-ranked non-declining examples:")
print(weak_picks[["baseline_rank", "content_id", "client_id", "baseline_refresh_score", "trend_direction", "reason_codes", "suggested_action_baseline"]].to_string(index=False))

print("\nLeakage checks:")
print("- Product flags used as features: no")
print("- Future-window columns used: no")
print("- Score components derive only from observed traffic, freshness, rank, and depth: yes")
print("- Any target-derived field included: no")


Top-50 declining rate: 0.34
Top-50 stable pages: 33

Weakest high-ranked non-declining examples:
 baseline_rank           content_id         client_id  baseline_refresh_score trend_direction                                                         reason_codes suggested_action_baseline
             2 content_4d1fe5b32dc2 client_19581e27de                0.934889          stable                      page_one_decay_risk|low_engagement_visible_page                   monitor
             3 content_07f2e7a6f38a client_19581e27de                0.934080          stable                      page_one_decay_risk|low_engagement_visible_page                   monitor
             4 content_e5ae436f9a16 client_4e07408562                0.933606          stable page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page    refresh_and_review_ctr
             5 content_3430a8b94511 client_19581e27de                0.933559          stable page_one_decay_risk|low_ctr_visible_page|low_engageme

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
